```sh
docker exec -it verl-qwen35 bash
cd /workspace/verl-train/verl 
python examples/data_preprocess/geo3k.py --local_save_dir /workspace/verl-train/data/geo3k
```
- infigui-g1
    - https://huggingface.co/datasets/InfiX-ai/omniact_grounding_filtered/viewer/default/train?row=0

### parquet

- EDA: 数据分析

```python
from datasets import load_datasets

# flat parquet
train_ds = load_dataset(
    "parquet",
    data_files="data/train.parquet",
)['train']
# flat parquet
test_ds = load_dataset(
    "parquet",
    data_files="data/test.parquet",
)['train']
```

```python
train_ds[:5]

# PIL.PngImagePlugin.PngImageFile
train_ds[0]['images'][0].save('test.png')

set(train_ds['data_source'])

train_ds.filter(lambda row: row['data_source'] == 'place_nav/w2p')
train_ds.filter(lambda row: row['data_source'] == 'place_nav/ts')
```

### tokenizer

`uv run --with ipython,torch,pandas,datasets,pillow,transformers,vllm ipython`

```python
from transformers import AutoTokenizer
MODEL_DIR = "/data1/models/Qwen3.5-4B"
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)

messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "计算 17 × 23，并给出答案。"},
]

# 默认开启
prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)


prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True,
)

# 显式关闭
prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)

# '<|im_start|>system\nYou are a helpful assistant.<|im_end|>\n<|im_start|>user\n计算 17 × 23，并给出答案。<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'

input_ids = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    enable_thinking=False,
)
```


```
# prompt，默认以及
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
计算 17 × 23，并给出答案。<|im_end|>
<|im_start|>assistant
<think>

# prompt，关闭 thinking 时
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
计算 17 × 23，并给出答案。<|im_end|>
<|im_start|>assistant
<think>

</think>
```

```python
import torch
from transformers import AutoTokenizer, AutoModelForImageTextToText

MODEL_DIR = "/data1/models/Qwen3.5-2B"
DEVICE = "cuda"

messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "中国的首都是哪里？"},
]

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_DIR,
    dtype=torch.bfloat16,
).to(DEVICE).eval()


# thinking
prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True,
)

inputs = tokenizer(
    prompt,
    return_tensors="pt",
).to(DEVICE)

prompt_length = inputs["input_ids"].shape[1]

with torch.inference_mode():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=1024,
        do_sample=False,
    )

response_ids = output_ids[0, prompt_length:]

generated_response = tokenizer.decode(
    response_ids,
    skip_special_tokens=False,
)
print(generated_response)

# no-thinking
prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)

inputs = tokenizer(
    prompt,
    return_tensors="pt",
).to(DEVICE)

prompt_length = inputs["input_ids"].shape[1]

with torch.inference_mode():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=1024,
        do_sample=False,
    )

response_ids = output_ids[0, prompt_length:]

generated_response = tokenizer.decode(
    response_ids,
    skip_special_tokens=False,
)
print(generated_response)
```

### rewards

- verl
    - `reward.custom_reward_function.path`
- slime
    - `--rm-type math`
        - slime/rollout/rm_hub/math_utils.py
    - `--custom-rm-path`
- cases
    - https://github.com/verl-project/verl/blob/main/verl/utils/reward_score/geo3k.py
    - https://github.com/verl-project/verl-recipe/blob/main/infigui-g1/reward_fn.py

### weights

```shell
source .venv/bin/activate
export HF_ENDPOINT=https://hf-mirror.com
export HF_HUB_DISABLE_XET=1
export HF_HUB_DOWNLOAD_TIMEOUT=300

hf download Qwen/Qwen2.5-VL-7B-Instruct \
  --local-dir models/Qwen2.5-VL-7B-Instruct
hf download Qwen/Qwen2.5-VL-3B-Instruct \
  --local-dir Qwen2.5-VL-3B-Instruct

# 原生多模态
# Dense
hf download Qwen/Qwen3.5-2B \
  --local-dir models/Qwen3.5-2B
hf download Qwen/Qwen3.5-4B \
  --local-dir models/Qwen3.5-4B
hf download Qwen/Qwen3.5-9B \
  --local-dir models/Qwen3.5-9B
# MOE
hf download Qwen/Qwen3.6-35B-A3B \
  --local-dir models/Qwen3.6-35B-A3B
```

- 建议将大的 model weights 存储在 nvme 的 ssd 上

### qwen3.5

| 模型 | `tie_word_embeddings` | 后果 |
|---|---|---|
| Qwen3.5-4B | `True` | `lm_head` 权重 = `embed` 权重（FSDP 管理，会被正常 onload）→ 融合核侥幸没事 |
| Qwen3.5-9B | `False` | `lm_head` 是独立权重，被 offload 后留在 CPU → 融合核 matmul `mat2 is on cpu` → 崩 |

- qwen 3.5 9b
    - use_fused_kernels=True & ref.fsdp_config.param_offload=True
        - 二者对于 qwen3.5 9b 是不兼容的
    - ref 的 lm_head/vocab 权重被 offload 到 CPU，而融合核直接拿这个权重去做 matmul → device 不匹配。
- fsdp 缺乏对 EP 的支持 （MoE）

### (swe) agent rollout

```python
messages = [
  # ── [0] system ── 只有格式约定，无工具 schema ──────────────────
  {"role": "system", "content":
   "You are a software engineer that can interact with a computer to solve coding tasks.\n\n"
   "IMPORTANT: Every response MUST follow this exact format:\n\n"
   "DISCUSSION\nYour reasoning about what to do next.\n\n"
   "```\nexactly_one_command_here\n```\n\n"
   "Rules:\n"
   "- Include EXACTLY ONE code block (``` ```) per response\n"
   "- The code block must be the LAST thing in your response\n"
   "- The code block contains the bash command or tool command to execute\n"
   "- Do NOT put example outputs or other code blocks in your response\n"
   "- You MUST actually edit files to fix the problem before submitting\n"
   "- NEVER run `submit` until you have made at least one code change ...\n"
   "- Running `submit` without making any edits will result in a FAILED submission"},

  # ── [1] user ── 任务 + 工具用法（唯一一处"工具文档"）────────────
  {"role": "user", "content":
   "\n"
   "I've uploaded a python code repository in the directory /train_6.\n\n"
   "<pr_description>\n"
   "The `remove_duplicates` function in `listutil.py` returns a set instead of\n"
   "a list. Fix it to return a list preserving the original order.\n"
   "</pr_description>\n\n"
   "Implement the necessary changes ... Do NOT modify any test files.\n\n"
   "You MUST follow these steps IN ORDER:\n"
   "1. Read and understand the existing code with `cat` and `ls`\n"
   "2. Identify exactly what needs to change\n"
   "3. Make the required code changes using `str_replace_editor` or bash commands\n"
   "   - `str_replace_editor` requires positional args: <command> <path> (no --path flag)\n"
   "   - Example: str_replace_editor str_replace /testbed/file.py --old_str \"<exact old>\" --new_str \"<new>\"\n"
   "   - Quote arguments carefully when strings contain spaces or newlines\n"
   "4. Verify your changes by reading the modified file\n"
   "5. Run `submit` ONLY AFTER you have made actual code changes\n\n"
   "CRITICAL: Do NOT run `submit` before editing files. ..."},

  # ── [2] assistant 轮1 ── 读文件 ───────────────────────────────
  {"role": "assistant", "content":
   "DISCUSSION\nThe task requires modifying the `remove_duplicates` function ...\n\n"
   "First, I will check the content of the file to understand the current implementation.\n\n"
   "```\ncat /train_6/listutil.py\n```"},

  # ── [3] user ── OBSERVATION：cat 的真实输出 ───────────────────
  {"role": "user", "content":
   "OBSERVATION:\ndef remove_duplicates(lst):\n    return set(lst)\n"},

  # ── [4] assistant 轮2 ── 第一次 str_replace_editor（参数位置错）─
  {"role": "assistant", "content":
   "DISCUSSION\nThe function ... currently returns a set ...\n\n"
   "```\nstr_replace_editor /train_6/listutil.py --old_str \"return set(lst)\" "
   "--new_str \"return list(dict.fromkeys(lst))\"\n```"},

  # ── [5] user ── OBSERVATION：工具报 usage 错误 ────────────────
  {"role": "user", "content":
   "OBSERVATION:\nusage: str_replace_editor [-h] ... command path\n"
   "str_replace_editor: error: the following arguments are required: path\n"},

  # ── [6] assistant 轮3 ── 换个顺序（还是错）───────────────────
  {"role": "assistant", "content":
   "DISCUSSION\nI made a mistake in the command syntax. ...\n\n"
   "```\nstr_replace_editor --old_str \"return set(lst)\" "
   "--new_str \"return list(dict.fromkeys(lst))\" /train_6/listutil.py\n```"},

  # ── [7] user ── OBSERVATION：同样的 usage 错误 ───────────────
  {"role": "user", "content":
   "OBSERVATION:\nusage: str_replace_editor ... \n"
   "str_replace_editor: error: the following arguments are required: path\n"},

  # ── [8] assistant 轮4 ── 终于对了 command+path 顺序 ──────────
  {"role": "assistant", "content":
   "DISCUSSION\n... The correct usage is `str_replace_editor <command> <path>` ...\n\n"
   "```\nstr_replace_editor str_replace /train_6/listutil.py "
   "--old_str \"return set(lst)\" --new_str \"return list(dict.fromkeys(lst))\"\n```"},

  # ── [9] user ── OBSERVATION：编辑成功 ────────────────────────
  {"role": "user", "content":
   "OBSERVATION:\nThe file /train_6/listutil.py has been edited. Here's the result "
   "of running `cat -n` ...\n     1\tdef remove_duplicates(lst):\n"
   "     2\t    return list(dict.fromkeys(lst))\nReview the changes ..."},

  # ── [10] assistant 轮5 ── 冗余：又发一遍同样的替换 ───────────
  {"role": "assistant", "content":
   "The function now correctly returns a list ...\n\n"
   "```\nstr_replace_editor str_replace /train_6/listutil.py "
   "--old_str \"return set(lst)\" --new_str \"return list(dict.fromkeys(lst))\"\n```"},

  # ── [11] user ── OBSERVATION：没得替换了（已改过）───────────
  {"role": "user", "content":
   "OBSERVATION:\nNo replacement was performed, old_str `return set(lst)` "
   "did not appear verbatim in /train_6/listutil.py.\n"},

  # ── [12] assistant 轮6 ── 重新 cat 确认 ──────────────────────
  {"role": "assistant", "content":
   "DISCUSSION\nThe issue is that the `old_str` must match exactly. ...\n\n"
   "```\ncat /train_6/listutil.py\n```"},

  # ── [13] user ── OBSERVATION：确认已是新代码 ────────────────
  {"role": "user", "content":
   "OBSERVATION:\ndef remove_duplicates(lst):\n    return list(dict.fromkeys(lst))\n"},

  # ── [14] assistant 轮7 ── 又一次冗余替换 ────────────────────
  {"role": "assistant", "content":
   "DISCUSSION\nThe function was already updated ...\n\n"
   "```\nstr_replace_editor str_replace /train_6/listutil.py "
   "--old_str \"return set(lst)\" --new_str \"return list(dict.fromkeys(lst))\"\n```"},

  # ── [15] user ── OBSERVATION：同上，没得替换 ───────────────
  {"role": "user", "content":
   "OBSERVATION:\nNo replacement was performed, old_str `return set(lst)` "
   "did not appear verbatim in /train_6/listutil.py.\n"},

  # ── [16] assistant 轮8 ── 提交 ──────────────────────────────
  {"role": "assistant", "content":
   "DISCUSSION\nThe `old_str` ... is no longer present ... the task is complete. ...\n\n"
   "```\nsubmit\n```"},

  # ── [17] user ── OBSERVATION：submit 返回最终 diff（→ reward）─
  {"role": "user", "content":
   "OBSERVATION:\ndiff --git a/listutil.py b/listutil.py\n"
   "index 303218f..041ff2c 100644\n--- a/listutil.py\n+++ b/listutil.py\n"
   "@@ -1,2 +1,2 @@\n def remove_duplicates(lst):\n"
   "-    return set(lst)\n+    return list(dict.fromkeys(lst))"},
]
```